## With manual log-in but automatic Chrome opening

In [ ]:
# ================================================================
# linkedin_scraper_heuristics.py
# Run: python linkedin_scraper_heuristics.py
# Requirements: pip install undetected-chromedriver selenium pandas tabulate
# ================================================================

import undetected_chromedriver as uc  # For automated Chrome browser that avoids detection
from selenium.webdriver.common.by import By  # For locating elements on page
import time, re, json, os  # Time for delays, re for regex, json/os for file handling
import pandas as pd  # For CSV and DataFrame handling
from tabulate import tabulate  # For printing tables nicely
from datetime import datetime  # For timestamped filenames

# ---------- EXPERIENCE PARSER CLASS ----------
class ExperienceParser:
    def __init__(self, driver):
        self.driver = driver

    def clean_text(self, txt):
        return re.sub(r"(See more|Mostra|Vedi).*", "", txt or "", flags=re.I).strip()

    def parse(self):
        experiences = []
        try:
            exp_section = self.driver.find_element(By.XPATH,
                "//section[contains(@id,'experience') or .//h2[contains(.,'Experience')]]")
            exp_items = exp_section.find_elements(By.XPATH, ".//li[contains(@class,'artdeco-list__item')]")

            for item in exp_items:
                # Check if this item is an attachment / certificate (skip if no role/company)
                try:
                    role_check = item.find_element(By.XPATH, ".//div[contains(@class,'t-bold')]/span").text.strip()
                    if not role_check:
                        continue  # Skip attachments
                except:
                    continue

                # Check for nested roles (multi-role job)
                subroles = item.find_elements(By.XPATH,
                    ".//ul/li[contains(@class,'bOjCJRCMPklZMZTMGFUpKfebsYzcUJpracU')]")
                
                # Extract company from top-level block
                try:
                    company = item.find_element(By.XPATH,
                        ".//div[contains(@class,'t-bold')]/span").text.strip()
                except:
                    company = ""

                # Determine if multi-role
                valid_subroles = []
                for sub in subroles:
                    try:
                        sub.find_element(By.XPATH, ".//div[contains(@class,'t-bold')]/span").text.strip()
                        valid_subroles.append(sub)  # Only keep subroles with role text
                    except:
                        continue

                if valid_subroles:
                    # Multi-role: parse each nested role
                    for sub in valid_subroles:
                        exp = self.extract_role(sub, parent_company=company)
                        experiences.append(exp)
                else:
                    # Single-role: parse top-level item
                    exp = self.extract_role(item)
                    experiences.append(exp)

        except Exception as e:
            print("⚠️ Experience parsing error:", e)

        return experiences

    def extract_role(self, node, parent_company=""):
        """Extract a single role block with robust duration/location detection."""
        # Role
        try:
            role = node.find_element(By.XPATH,
                ".//div[contains(@class,'t-bold')]/span").text.strip()
        except:
            role = ""

        # Company
        try:
            company = node.find_element(By.XPATH,
                ".//span[contains(@class,'t-14') and not(contains(@class,'t-black--light'))]/span").text.strip()
        except:
            company = parent_company

        # Duration (renamed from Time) and Location
        duration = ""
        location_exp = ""
        light_spans = node.find_elements(By.XPATH,
            ".//span[contains(@class,'t-14') and contains(@class,'t-black--light')]/span")
        
        for span in light_spans:
            txt = span.text.strip()
            if re.search(r"\d{4}", txt):  # Looks like a date -> duration
                duration = txt
            elif "," in txt or "·" in txt:  # Looks like location
                location_exp = txt

        # Optional Description
        try:
            desc = node.find_element(By.XPATH,
                ".//div[contains(@class,'inline-show-more-text')]//span[@aria-hidden='true']").text.strip()
        except:
            desc = ""

        return {
            "Role": self.clean_text(role),
            "Company": self.clean_text(company),
            "Location": self.clean_text(location_exp),
            "Duration": self.clean_text(duration),
            "Description": self.clean_text(desc)
        }
    
# ---------- EDUCATION PARSER CLASS (no Field) ----------
class EducationParser:
    def __init__(self, driver):
        self.driver = driver

    def clean_text(self, txt):
        return re.sub(r"(See more|Mostra|Vedi).*", "", txt or "", flags=re.I).strip()

    def parse(self):
        education_list = []
        try:
            edu_section = self.driver.find_element(By.XPATH,
                "//section[contains(@id,'education') or .//h2[contains(.,'Education')]]")
            edu_items = edu_section.find_elements(By.XPATH, ".//li")

            for item in edu_items:
                lines = [self.clean_text(l) for l in item.text.split("\n") if self.clean_text(l)]
                school, degree, time_text = "", "", ""

                for l in lines:
                    # Extract duration if it contains a year
                    if re.search(r"\d{4}", l):
                        time_text = l
                    # Degree heuristics
                    elif any(x.lower() in l.lower() for x in ["bachelor", "master", "phd", "laurea"]):
                        degree = l
                    # First line is usually school
                    elif not school:
                        school = l

                education_list.append({
                    "School": school,
                    "Degree": degree,
                    "Time": time_text
                })

            # Optional: sort by start year if duration is present
            def get_start_year(edu):
                match = re.search(r"(\d{4})", edu["Time"])
                return int(match.group(1)) if match else 0

            education_list.sort(key=get_start_year, reverse=True)

        except Exception as e:
            print("⚠️ Education parsing error:", e)

        return education_list

# ---------- CONFIG ----------
PROFILE_URL = "https://www.linkedin.com/in/eliana-di-lodovico-570171192"  # LinkedIn profile URL
CHROMEDRIVER_PATH = r"C:\Users\Utente\Downloads\chromedriver-win64\chromedriver-win64\chromedriver.exe"  # Path to ChromeDriver
WAIT_TIMEOUT = 300   # Seconds to wait for manual login

# ---------- LAUNCH BROWSER ----------
options = uc.ChromeOptions()  # Initialize Chrome options
options.add_argument("--start-maximized")  # Start browser maximized
driver = uc.Chrome(driver_executable_path=CHROMEDRIVER_PATH, options=options)  # Launch browser
driver.get("https://www.linkedin.com/login")  # Open LinkedIn login page
print("Please log in manually in the browser window...")  # Prompt user

# Wait for login
start = time.time()  # Track starting time
while True:
    time.sleep(1)  # Wait 1 second between checks
    if any(c.get("name") == "li_at" for c in driver.get_cookies()):  # Check for LinkedIn login cookie
        print("✅ Login detected — proceeding.")  # Login successful
        break
    if time.time() - start > WAIT_TIMEOUT:  # Timeout reached
        input("⚠️ Timeout reached. Press Enter if already logged in.")  # Allow manual continuation
        break

# ---------- NAVIGATE TO PROFILE ----------
driver.get(PROFILE_URL)  # Open target LinkedIn profile
time.sleep(2)  # Wait for page to load

# ---------- UTILITIES ----------
def safe_click(xpath):
    """Safely click an element if it exists."""
    try:
        el = driver.find_element(By.XPATH, xpath)  # Locate element
        driver.execute_script("arguments[0].scrollIntoView(true);", el)  # Scroll into view
        time.sleep(0.3)  # Short wait
        el.click()  # Click element
        time.sleep(0.7)  # Wait for action to complete
        return True  # Click successful
    except:
        return False  # Element not found / click failed

def clean_line(line):
    """Clean a line of text, remove 'See more' or similar."""
    line = re.sub(r"(See more|Mostra|Vedi).*", "", line, flags=re.I)  # Remove common LinkedIn expansion labels
    return line.strip()  # Remove leading/trailing spaces

# Expand buttons for extra content
expanders = [
    "//button[contains(.,'See more')]",
    "//button[contains(.,'Mostra')]",
    "//button[contains(@aria-label,'See more')]",
    "//button[contains(.,'Show all education')]",
    "//button[contains(.,'Show all experiences')]"
]
for xp in expanders:
    safe_click(xp)  # Try clicking each "see more" button

# Scroll page to load dynamic content
for f in [0.25,0.5,0.75,1.0]:
    driver.execute_script(f"window.scrollTo(0, document.body.scrollHeight*{f});")  # Scroll fractionally
    time.sleep(0.7)  # Wait for content to load

# ---------- BASIC INFO ----------
def try_selectors(selectors):
    """Try multiple CSS/XPath selectors and return first non-empty text."""
    for by, sel in selectors:
        try:
            el = driver.find_element(by, sel)
            txt = el.text.strip()
            if txt:
                return txt
        except:
            pass
    return None

name = try_selectors([(By.CSS_SELECTOR,"h1.text-heading-xlarge"),(By.XPATH,"//main//h1")])  # Profile name
headline = try_selectors([(By.CSS_SELECTOR,"div.text-body-medium.break-words")])  # Headline
location = try_selectors([(By.CSS_SELECTOR,"span.text-body-small.inline.t-black--light.break-words")])  # Location
about = try_selectors([(By.XPATH,"//section[contains(@class,'pv-about-section')]//p")])  # About section

# ---------- EXPERIENCE ----------
parser = ExperienceParser(driver)
experience = parser.parse()

# ---------- EDUCATION ----------
edu_parser = EducationParser(driver)
education = edu_parser.parse()

# ---------- PRINT TABLES ----------
print("\n=== LINKEDIN PROFILE ===")
print(f"👤 Name: {name}")
print(f"💼 Headline: {headline}")
print(f"📍 Location: {location}")
if about:
    print(f"📝 About: {about}")

if experience:
    print("\n=== EXPERIENCE ===")
    exp_rows = [[e.get("Role",""), e.get("Company",""), e.get("Location",""), e.get("Duration",""), e.get("Description","")] 
            for e in experience]
    exp_headers = ["Role","Company","Location","Duration","Description"]

    print(tabulate(exp_rows, headers=exp_headers, tablefmt="fancy_grid"))  # Pretty table output

if education:
    print("\n=== EDUCATION ===")
    edu_rows = [[e.get("School",""),e.get("Degree",""),e.get("Time","")] for e in education]
    edu_headers = ["School","Degree","Time"]
    print(tabulate(edu_rows, headers=edu_headers, tablefmt="fancy_grid"))  # Pretty table output

# ---------- SAVE JSON & CSV ----------
ts = datetime.now().strftime("%Y%m%d_%H%M%S")  # Timestamp for unique filenames
json_file = f"linkedin_profile_{ts}.json"
csv_exp = f"experience_{ts}.csv"
csv_edu = f"education_{ts}.csv"
csv_basic = f"basic_{ts}.csv"

with open(json_file,"w",encoding="utf-8") as f:  # Save full profile as JSON
    json.dump({"Name":name,"Headline":headline,"Location":location,"About":about,
               "Experience":experience,"Education":education}, f, ensure_ascii=False, indent=2)

pd.DataFrame(experience).to_csv(csv_exp,index=False)  # Save experience as CSV
pd.DataFrame(education).to_csv(csv_edu,index=False)  # Save education as CSV
pd.DataFrame([{"Name":name,"Headline":headline,"Location":location,"About":about}]).to_csv(csv_basic,index=False)  # Save basic info CSV

print(f"\n✅ JSON saved to {json_file}")
print(f"✅ Experience CSV saved to {csv_exp}")
print(f"✅ Education CSV saved to {csv_edu}")
print(f"✅ Basic info CSV saved to {csv_basic}")

driver.quit()  # Close browser